# E2 en Google Colab: Moirai con ajuste fino del cabezal (RQ2, sección 14.2)

Sesión aparte (su biblioteca pide otras versiones). Orden: representación congelada de cada ventana con los pesos
oficiales (`src/embeber.py`, se guarda en el disco local `/content/emb`, no en Drive) → ajuste previo y fase 1 del
cabezal (`busqueda_arq.py`, 1 000 arquitecturas, sin muestra) → fase 2 → corridas finales. Todo en la MISMA sesión:
si Colab se desconecta, los resultados ya guardados en Drive se conservan y la representación se vuelve a calcular.

**Cada vez:** Entorno de ejecución → ejecutar todas las celdas. Betzaida acepta el permiso de Google Drive.

## 1. Qué correr (editar solo esta celda)

In [ ]:
TAREA = "cola_Moirai"
MODELOS = ["Moirai"]
K = 10
ORIGENES = "ultimo"
COLA = [
    "src/embeber.py Moirai",
    "src/busqueda_arq.py Moirai",
    "src/fase2.py --k 10 --origenes ultimo Moirai",
    "src/finales.py Moirai",
]
AUTO_DESCONECTAR = True

## 2. GPU asignada

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 3. Google Drive, datos y código

In [ ]:
import os, subprocess, zipfile
from google.colab import drive
drive.mount("/content/drive")
BASE = "/content/drive/MyDrive/DL_viento"
ZIP = "/content/drive/MyDrive/DL_viento_colab.zip"
if not os.path.exists(f"{BASE}/datos/panel_multivariado_causal.csv"):
    assert os.path.exists(ZIP), "Falta subir DL_viento_colab.zip a la raíz de Google Drive"
    zipfile.ZipFile(ZIP).extractall(BASE)
    print("Datos y resultados descomprimidos en", BASE)
os.makedirs(f"{BASE}/logs", exist_ok=True)

REPO = "/content/repo"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "e2",
                    "https://github.com/BetzyW/Deep-Learning-Wind.git", REPO], check=True)
else:
    # solo se actualiza el código (src/): resultados/ es un enlace a Drive y un pull completo chocaría con él
    subprocess.run(["git", "-C", REPO, "fetch", "-q", "origin", "e2"], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", "-q", "origin/e2", "--", "src"], check=True)
# datos y resultados apuntan a Drive: todo lo que se calcula queda guardado aunque Colab se desconecte
for enlace, destino in [("datos", f"{BASE}/datos"), ("resultados/e2", f"{BASE}/resultados/e2")]:
    ruta = f"{REPO}/{enlace}"
    if os.path.islink(ruta):
        continue
    subprocess.run(["rm", "-rf", ruta], check=True)
    os.makedirs(os.path.dirname(ruta), exist_ok=True)
    os.symlink(destino, ruta)
print(subprocess.run(["git", "-C", REPO, "log", "-1", "--format=Código: %h %s", "origin/e2"], capture_output=True, text=True).stdout)

## 4. Versiones (se reportan junto con el hardware)

In [ ]:
# uni2ts fija numpy 1.26 y scipy 1.11, que no compilan en el Python 3.13 de Colab: se instala sin sus dependencias
# y se agregan solo las que necesita para importar Moirai (probado en el PC con numpy 2.4.6 y torch 2.14)
!pip install -q --no-deps uni2ts==2.0.0 einops==0.7.0 jaxtyping wadler_lindig gluonts hydra-core omegaconf antlr4-python3-runtime==4.9.3 python-dotenv lightning lightning-utilities torchmetrics toolz safetensors
# mismas versiones de numpy, pandas y scipy que en el PC (requirements.txt); torch se deja el de Colab (CUDA)
!pip install -q numpy==2.4.6 pandas==3.0.6 scipy==1.17.1
import torch, numpy, pandas, platform
print("Python", platform.python_version(), "| torch", torch.__version__, "| CUDA", torch.version.cuda,
      "| numpy", numpy.__version__, "| pandas", pandas.__version__)
# si numpy o pandas cambiaron de versión, Colab pide reiniciar la sesión: hacerlo y volver a ejecutar todo


## 5. Correr

La salida se guarda también en `MyDrive/DL_viento/logs/`. Se puede cerrar el navegador (Colab Pro+).

In [ ]:
import time
log = f"{BASE}/logs/colab_{TAREA}_{time.strftime('%Y%m%d_%H%M')}.log"
if TAREA == "fase1":
    comandos = [["python", "-u", "src/busqueda_arq.py", *MODELOS]]
elif TAREA == "fase2":
    comandos = [["python", "-u", "src/fase2.py", "--k", str(K), "--origenes", ORIGENES, *MODELOS]]
else:
    comandos = [["python", "-u", *c.split()] for c in COLA]
with open(log, "w") as f:
    for cmd in comandos:
        print(" ".join(cmd), "→", log)
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for linea in proc.stdout:
            print(linea, end=""); f.write(linea); f.flush()
        print("Terminó con código", proc.wait(), flush=True)
if AUTO_DESCONECTAR:
    from google.colab import runtime
    runtime.unassign()   # cada hora de GPU encendida cuesta unidades aunque no calcule

## 6. Avance (se puede correr en otra sesión mientras la celda 5 trabaja)

In [ ]:
import glob, pandas as pd
for ruta in sorted(glob.glob(f"{BASE}/resultados/e2/busqueda/*_fase*.csv")):
    d = pd.read_csv(ruta)
    print(f"{os.path.basename(ruta):24s} {len(d):5d} filas | mejor RMSE {d.rmse_val.min():.4f} | "
          f"{d.segundos.mean():.0f} s de media")

## 7. Traer los resultados al PC

Descargar de Drive la carpeta `DL_viento/resultados/e2/busqueda` y, en el PC, desde la carpeta del repositorio:

`python src/sincronizar.py unir RUTA_DE_LA_CARPETA_DESCARGADA`

Solo agrega las filas nuevas, así que se puede repetir sin duplicar.